# BirdNET-Pi Species Stats

Local offline equivalent of the BirdNET-Pi statistics dashboard.
Connects directly to your SQLite database and recording files — no server required.

**Setup:** `pip install -r ../requirements-notebook.txt`

**Usage:** Edit the paths below, then `Kernel → Restart & Run All`.

In [1]:
import os

# ── Edit these paths to match your setup ──────────────────────────────────────
DB_PATH        = os.path.expanduser('/mnt/nas/sibley_backup/peterson/db/birds.db')
RECORDINGS_DIR = os.path.expanduser('/mnt/nas/sibley_backup/peterson/recordings/Extracted')
LATITUDE       = 40.1095847    # decimal degrees, for sunrise/sunset overlay
LONGITUDE      = -75.1965847
# ─────────────────────────────────────────────────────────────────────────────

if not os.path.exists(DB_PATH):
    raise FileNotFoundError(f'Database not found: {DB_PATH}')
if not os.path.isdir(RECORDINGS_DIR):
    print(f'Warning: recordings directory not found: {RECORDINGS_DIR}')
print('Config OK')

Config OK


In [2]:
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, Image, Audio, clear_output
import plotly.io as pio

sys.path.insert(0, os.path.dirname(os.path.abspath('helpers.py')))
from helpers import (
    get_connection, load_data,
    apply_date_filter, apply_resample, RESAMPLE_MAP,
    hms_to_dec, hms_to_str, sunrise_sunset_scatter,
    polar_trace, POLAR_LAYOUT,
    recording_path,
)

pio.templates.default = 'plotly_white'

conn   = get_connection(DB_PATH)
df_all = load_data(conn)
print(f'Loaded {len(df_all):,} detections across {df_all["Com_Name"].nunique()} species')
print(f'Date range: {df_all.index.min().date()} \u2192 {df_all.index.max().date()}')

Loaded 2,037 detections across 29 species
Date range: 2026-03-31 → 2026-04-02


In [3]:
from datetime import timedelta
from sklearn.preprocessing import normalize
from numpy import ma

min_date = df_all.index.min().date()
max_date = df_all.index.max().date()
all_species = [str(s) for s in df_all['Com_Name'].value_counts().index]

w_single_day = widgets.Checkbox(description='Single Day View', value=False)
w_date       = widgets.DatePicker(description='Date', value=max_date, continuous_update=False)
w_start      = widgets.DatePicker(description='Start', value=min_date, continuous_update=False)
w_end        = widgets.DatePicker(description='End',   value=max_date, continuous_update=False)
w_resample   = widgets.RadioButtons(options=list(RESAMPLE_MAP), value='15 minutes', description='Resample:')
w_topn       = widgets.IntSlider(description='Top N', min=1, max=min(50, len(all_species)), value=10, continuous_update=False)
w_species    = widgets.Dropdown(description='Species', options=['All'] + all_species, value='All')
w_colorscale = widgets.Dropdown(description='Heatmap pal', options=px.colors.named_colorscales(), value='blues')

w_recording = None
rec_container = widgets.VBox()
out_charts = widgets.Output()
out_audio  = widgets.Output()
date_box   = widgets.VBox([w_start, w_end])


def toggle_single_day(change=None):
    """Switch between single-day and date-range controls."""
    if w_single_day.value:
        date_box.children = [w_date]
        # Remove 'Daily' from resample options in single-day mode
        opts = [k for k in RESAMPLE_MAP if k != 'Daily']
        if w_resample.value not in opts:
            w_resample.value = '15 minutes'
        w_resample.options = opts
    else:
        date_box.children = [w_start, w_end]
        w_resample.options = list(RESAMPLE_MAP)

toggle_single_day()
w_single_day.observe(toggle_single_day, names='value')


def show_recording(change=None):
    """Show spectrogram and audio for the currently selected recording."""
    with out_audio:
        clear_output(wait=True)
        if w_recording is None:
            return
        label = w_recording.value
        if label is None and w_recording.options:
            first = w_recording.options[0]
            label = first[1] if isinstance(first, tuple) else first
        if not label or not label.strip():
            return
        filename = label.rsplit('\u2014 ', 1)[-1].strip()
        row = df_all[df_all['File_Name'] == filename]
        if row.empty:
            print(f'Recording not in database: {filename}')
            return
        audio_path, png_path = recording_path(row.iloc[0], RECORDINGS_DIR)
        if os.path.exists(png_path):
            display(Image(filename=png_path, width=600))
        else:
            print(f'Spectrogram not found:\n  {png_path}')
        if os.path.exists(audio_path):
            display(Audio(filename=audio_path))
        else:
            print(f'Recording not found:\n  {audio_path}')


def render(change=None):
    global w_recording

    # Resolve date range
    if w_single_day.value:
        start = end = w_date.value
    else:
        start, end = w_start.value, w_end.value
    if not start or not end or start > end:
        return

    df = apply_date_filter(df_all, start, end)
    if df.empty:
        with out_charts:
            clear_output()
            print('No detections in selected date range.')
        rec_container.children = [out_audio]
        return

    df5       = apply_resample(df, w_resample.value)
    counts    = df5.value_counts()
    hourly    = pd.crosstab(df5, df5.index.hour, dropna=True, margins=True)
    top_n     = min(w_topn.value, len(counts))
    selected  = str(w_species.value) if w_species.value else None
    is_all    = (selected == 'All')

    # ── Recording dropdown (only for specific species) ──
    if not is_all and selected:
        recs_sorted = df[df['Com_Name'] == selected].sort_index(ascending=False)
        rec_opts = [
            f"{r.Index.strftime('%Y-%m-%d %H:%M')} ({r.Confidence*100:.0f}%) \u2014 {r.File_Name}"
            for r in recs_sorted.itertuples()
        ]
        w_recording = widgets.Dropdown(
            description='Recording',
            options=rec_opts if rec_opts else [''],
        )
        w_recording.observe(show_recording, names='value')
        rec_container.children = [w_recording, out_audio]
        show_recording()
    else:
        w_recording = None
        rec_container.children = []
        with out_audio:
            clear_output(wait=True)

    # ── Single day view ──
    if w_single_day.value:
        _render_single_day(df, df5, counts, top_n, start)
        return

    # ── Multi-day views ──
    with out_charts:
        clear_output(wait=True)

        if is_all:
            # Top-N bar chart
            go.Figure(
                go.Bar(y=counts[:top_n].index.tolist(), x=counts[:top_n].values.tolist(),
                       orientation='h', marker_color='seagreen')
            ).update_layout(
                title=f'Top {top_n} Species  {start} \u2192 {end}',
                yaxis={'categoryorder': 'total ascending'},
                margin=dict(l=0, r=0, t=40, b=0),
                height=max(300, top_n * 25)
            ).show()

            if 'All' in hourly.index:
                # Polar for all detections
                go.Figure(polar_trace(hourly.loc['All'])).update_layout(
                    title=f'All species \u2014 detections by hour  {start} \u2192 {end}',
                    polar=POLAR_LAYOUT, height=450
                ).show()

                # Daily bar for all
                daily_ct = pd.crosstab(df5, df5.index.date, dropna=True, margins=True)
                if 'All' in daily_ct.index:
                    row = daily_ct.loc['All']
                    go.Figure(
                        go.Bar(x=[str(c) for c in daily_ct.columns[:-1]],
                               y=row.iloc[:-1].tolist(), marker_color='seagreen')
                    ).update_layout(
                        title=f'All species \u2014 daily detections  Total: {int(hourly.loc["All", "All"]):,}',
                        height=350
                    ).show()
            return

        # ── Specific species, multi-day ──
        if selected not in hourly.index:
            print(f'No detections for {selected} in this date range.')
            return

        # Polar chart
        go.Figure(polar_trace(hourly.loc[selected])).update_layout(
            title=f'{selected} \u2014 detections by hour  {start} \u2192 {end}',
            polar=POLAR_LAYOUT, height=450
        ).show()

        if w_resample.value == 'Daily':
            # Heatmap view (DAILY resample mode)
            _render_heatmap(df, selected, start, end)
        else:
            # Heatmap always shown for multi-day
            _render_heatmap(df, selected, start, end)

            # Daily bar chart
            daily_ct = pd.crosstab(df5, df5.index.date, margins=True)
            if selected in daily_ct.index:
                row   = daily_ct.loc[selected]
                sp_df = df[df['Com_Name'] == selected]
                go.Figure(
                    go.Bar(x=[str(c) for c in daily_ct.columns[:-1]],
                           y=row.iloc[:-1].tolist(), marker_color='seagreen')
                ).update_layout(
                    title=(f'{selected} \u2014 daily detections  '
                           f'Total: {int(hourly.loc[selected, "All"]):,}  '
                           f'Max conf: {sp_df["Confidence"].max()*100:.1f}%  '
                           f'Median: {sp_df["Confidence"].median()*100:.1f}%'),
                    height=350
                ).show()


def _render_heatmap(df, selected, start, end):
    """Time-of-day heatmap with sunrise/sunset overlay for a specific species."""
    sp = df['Com_Name'][df['Com_Name'] == selected].resample('15min').count()
    sp_mi = pd.Series(sp.values, index=pd.MultiIndex.from_arrays([sp.index.date, sp.index.time]))
    pivot = sp_mi.unstack().fillna(0)
    fig_dec = [hms_to_dec(t) for t in pivot.columns]
    fig_str = [hms_to_str(t) for t in pivot.columns]
    step    = max(1, len(fig_str) // 12)
    pivot.columns = fig_dec
    x_ss, y_ss, t_ss = sunrise_sunset_scatter(pivot.index.tolist(), LATITUDE, LONGITUDE)
    go.Figure(data=[
        go.Heatmap(x=[d.strftime('%d-%m-%Y') for d in pivot.index],
                   y=fig_dec, z=pivot.values.T.tolist(),
                   colorscale=w_colorscale.value, showscale=False),
        go.Scatter(x=x_ss, y=y_ss, mode='lines', hoverinfo='text',
                   text=t_ss, line_color='orange', line_width=1, name=' '),
    ]).update_layout(
        title=f'{selected} \u2014 detection heatmap  {start} \u2192 {end}',
        yaxis=dict(tickmode='array', tickvals=fig_dec[::step], ticktext=fig_str[::step]),
        height=500
    ).show()


def _render_single_day(df, df5, counts, top_n, date):
    """Single day view: top-N bar + species-by-hour heatmap side by side."""
    with out_charts:
        clear_output(wait=True)

        resample_label = w_resample.value

        # Top-N bar chart
        top_species = counts[:top_n]
        go.Figure(
            go.Bar(y=top_species.index.tolist(), x=top_species.values.tolist(),
                   orientation='h', marker_color='seagreen')
        ).update_layout(
            title=f'Top {top_n} Species for {date}',
            yaxis={'categoryorder': 'total ascending'},
            margin=dict(l=0, r=0, t=40, b=0),
            height=max(300, top_n * 25)
        ).show()

        # Species-by-hour heatmap (top N species, normalized per species)
        df6 = df5.to_frame(name='Com_Name') if not isinstance(df5, pd.DataFrame) else df5
        freq_order = list(df6['Com_Name'].value_counts().iloc[:top_n].index)

        df6 = df6.copy()
        df6['Hour of Day'] = [r.hour for r in df6.index.time]
        heat = pd.crosstab(df6['Com_Name'], df6['Hour of Day'])
        # Keep only top N species
        heat = heat.reindex(freq_order).fillna(0)

        hours_in_day = pd.Series(data=range(0, 24))
        heat_frame = pd.DataFrame(data=0, index=heat.index, columns=hours_in_day)
        heat = (heat + heat_frame).fillna(0)
        heat_normalized = normalize(heat.values, axis=1, norm='l1')

        labels = heat.values.astype(int).astype('str')
        labels[labels == '0'] = ''

        go.Figure(
            go.Heatmap(x=heat.columns.tolist(), y=heat.index.tolist(),
                       z=heat_normalized, showscale=False,
                       text=labels, texttemplate='%{text}', colorscale='Blugrn')
        ).update_layout(
            title=f'{date} Detections on {resample_label} interval',
            yaxis=dict(autorange='reversed'),
            height=max(300, top_n * 30)
        ).show()


for w in (w_single_day, w_date, w_start, w_end, w_resample, w_topn, w_colorscale, w_species):
    w.observe(render, names='value')

display(
    widgets.HBox([
        widgets.VBox([w_single_day, date_box, w_resample, w_topn]),
        widgets.VBox([w_species, rec_container, w_colorscale]),
    ]),
    out_charts,
)
render()


Output()